## 6. raw_to_zarr compatibility (echopype)

The echopype-based `raw_to_zarr` path loads the entire file into memory at once
(`ep.open_raw` + `ep.calibrate.compute_Sv`), which OOMs on these 1 GB files.

Possible workarounds:
- Split each .raw file into smaller chunks before conversion (echopype doesn't support streaming)
- Use `load_raw()` directly with `max_pings` to process in batches
- Run conversion on a machine with more RAM (each 1 GB file needs ~8–12 GB for calibration)
- Use the CRIMAC Docker preprocessor instead, which handles memory more carefully

In [ ]:
fig, axes = plt.subplots(1, len(raw_files), figsize=(5 * len(raw_files), 6))
if len(raw_files) == 1:
    axes = [axes]

for ax, f in zip(axes, raw_files):
    try:
        data = load_raw(str(f), frequency=38000, max_pings=200)
        vmin, vmax = data.default_vmin_vmax
        ax.imshow(data.sv_db, aspect='auto', cmap='viridis',
                  vmin=vmin, vmax=vmax,
                  extent=[0, data.n_pings, data.max_range, 0])
        ax.set_title(f.name[:30], fontsize=9)
        ax.set_xlabel('Ping')
        ax.set_ylabel('Range (m)')
    except Exception as e:
        ax.text(0.5, 0.5, f"FAILED:\n{e}", transform=ax.transAxes,
                ha='center', va='center', fontsize=8, color='red')
        ax.set_title(f.name[:30], fontsize=9)

plt.tight_layout()
plt.show()

## 5. Quick echogram preview

Visualise the first 200 pings of the 38 kHz channel from each file to confirm the data looks reasonable.

In [ ]:
# Test each file with load_raw — limited to 200 pings to keep memory low.
# The 2021 file works out of the box; the 2019 files need the SoundVelocity fix.
for f in raw_files:
    print(f"=== {f.name} ===")
    try:
        data = load_raw(str(f), frequency=38000, max_pings=200)
        print(f"  OK — channel={data.channel_id}, freq={data.frequency_hz} Hz")
        print(f"  shape={data.sv_db.shape}, range={data.max_range:.1f} m")
        print(f"  sound_velocity={data.sound_velocity}, sample_interval={data.sample_interval}")
        if data.gps is not None and not data.gps.empty:
            print(f"  GPS: {len(data.gps)} fixes")
        else:
            print(f"  GPS: none")
    except Exception as e:
        print(f"  FAILED: {e}")
    print()

## 4. Test load_raw (memory-safe, single channel, limited pings)

`load_raw` works on these files because it only loads one channel at a time and we can
cap the number of pings with `max_pings`. The `raw_to_zarr` / echopype path loads
everything at once, which is why it OOMs on 1 GB files.

In [ ]:
import matplotlib.pyplot as plt

# Parse GPS from the 2021 file (the only one with position data)
f_2021 = [f for f in raw_files if f.name.startswith("D2021")][0]
all_nme0 = []
with open(f_2021, 'rb') as fh:
    for _ in range(MAX_CHUNKS):
        chunk = fh.read(CHUNK_SIZE)
        if not chunk:
            break
        all_nme0.extend(gps_parser.parse_nme0_datagrams(chunk))

gps_df = gps_parser.parse_gps_sentences(all_nme0)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(gps_df['lon'], gps_df['lat'], '.-', markersize=4)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title(f'{f_2021.name} — GPS track ({len(gps_df)} fixes)')
ax.ticklabel_format(useOffset=False)
plt.tight_layout()
plt.show()

print(f"Centre: {gps_df['lat'].mean():.4f}°N, {gps_df['lon'].mean():.4f}°E")

## 3. Plot GPS track (2021 file)

The 2021 file was recorded at ~56.85°N, 5.53°E (North Sea, off western Denmark).

In [ ]:
CHUNK_SIZE = 50_000_000  # 50 MB chunks
MAX_CHUNKS = 5           # scan first 250 MB per file

for f in raw_files:
    print(f"=== {f.name} ===")
    all_nme0 = []
    with open(f, 'rb') as fh:
        for _ in range(MAX_CHUNKS):
            chunk = fh.read(CHUNK_SIZE)
            if not chunk:
                break
            all_nme0.extend(gps_parser.parse_nme0_datagrams(chunk))
    
    print(f"  NME0 records (first {MAX_CHUNKS * CHUNK_SIZE // 1e6:.0f} MB): {len(all_nme0)}")
    
    if all_nme0:
        # Show raw sentence types
        types = set()
        for r in all_nme0:
            s = r.get('sentence', '')
            if s:
                types.add(s.split(',')[0])
        print(f"  Sentence types: {types}")
        
        gps_df = gps_parser.parse_gps_sentences(all_nme0)
        if not gps_df.empty:
            print(f"  GPS fixes: {len(gps_df)}")
            print(f"  Lat: {gps_df['lat'].min():.6f} to {gps_df['lat'].max():.6f}")
            print(f"  Lon: {gps_df['lon'].min():.6f} to {gps_df['lon'].max():.6f}")
            print(f"  Time span: {gps_df['timestamp'].min()} → {gps_df['timestamp'].max()}")
        else:
            print("  No valid GPS position fixes (no GGA/RMC/GLL sentences)")
    print()

## 2. GPS location

The 2019 files only have `$SDVLW` (distance-log) sentences — no position data.
The 2021 file has GPGGA fixes. We scan in chunks to avoid loading the full 1 GB file into memory.

In [ ]:
for f in raw_files:
    print(f"=== {f.name} ===")
    
    # Read just 2 MB for datagram survey
    with open(f, 'rb') as fh:
        header = fh.read(2_000_000)
    
    types = raw_parser.find_datagram_types(header)
    for t, offsets in sorted(types.items()):
        print(f"  {t}: {len(offsets)} datagrams (first 2 MB)")
    
    # Channel settings — may fail on 2019 files
    try:
        settings = metadata.get_channel_settings(str(f))
        print(f"  Channels ({len(settings)}):")
        for ch in settings:
            freq = ch.get('frequency', '?')
            sid = ch.get('channel_id', '?')
            sc = ch.get('sample_count', '?')
            sv = ch.get('sound_velocity', 'MISSING')
            print(f"    {sid}  freq={freq} Hz  samples={sc}  sound_vel={sv}")
    except Exception as e:
        print(f"  *** get_channel_settings FAILED: {e}")
        # Show raw self-closing Channel elements to see what's missing
        for m in re.finditer(rb'<Channel\b([^>]+)/>', header[:500_000]):
            attrs = m.group(1).decode('ascii', errors='replace')
            print(f"    Raw attrs: {attrs[:150]}...")
            break  # just show one
    print()

## 1. Datagram types and channel metadata

Scan just the first 2 MB of each file for datagram types and channel configuration.
The 2019 files (`2019703_st126`) have 5 channels but lack `SoundVelocity` in their ping-config XML,
which causes `metadata.get_channel_settings()` to fail.

In [ ]:
import sys, os, re, struct
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.insert(0, '../..')
from raw_sensor_integration import raw_parser, metadata, gps_parser, ping_parser
from raw_sensor_integration.data_loader import load_raw

RAW_DIR = Path.home() / 'Downloads' / 'diffraw'
raw_files = sorted(RAW_DIR.glob("*.raw"))
print(f"{len(raw_files)} raw files, total {sum(f.stat().st_size for f in raw_files) / 1e9:.1f} GB")
for f in raw_files:
    print(f"  {f.name}  ({f.stat().st_size / 1e9:.2f} GB)")

# Compatibility check: diffraw files

These raw files come from a different source and have more frequencies than the usual dataset.
This notebook checks whether they work with the existing raw→zarr pipeline and inspects their contents.